# 07 — Protocolo final de evaluación temporal

Después de seleccionar LightGBM en el 04, adoptar su propio Platt en el 05 y diagnosticar estabilidad en el 06, fijamos las decisiones del experimento antes de puntuar test. La congelación es técnica y académica: conservamos costos hipotéticos y declaramos las limitaciones. No acredita un aval docente ni una estimación de gastos bancarios.

Este notebook verifica la cadena 03–06 y registra predictor, preparación, calibrador, políticas, métricas, escenarios y población futura. No ajusta estimadores ni genera probabilidades de test. Las funciones están en `src/fraud_cost/final_protocol.py` y el informe en `docs/07_protocolo_final.md`.


## 1. Evidencia necesaria

Verificamos los artefactos propios antes de deserializarlos. Recuperar un modelo por su nombre es insuficiente: debe corresponder a la selección, la representación y el calibrador que produjeron la evidencia. Las etapas 05/06 deben estar completas y sus tablas reconciliadas. Una reconstrucción aceptada conserva su recibo y sus hashes propios.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir() and (p / "notebooks").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from fraud_cost.final_protocol import (
    freeze_protocol, verify_frozen_protocol, verify_before_test,
    PROTOCOL, RECORD, AUDIT, DECISIONS,
)
from fraud_cost.artifacts import sha256_file

print("Etapa 07: congelación técnica; test todavía sin puntuar.")


Etapa 07: congelación técnica; test todavía sin puntuar.


## 2. Congelación y comprobaciones

El 04 eligió LightGBM referencia por AP media, con igual peso por ventana. El 05 adoptó el ajuste completo coincidente del 03 y su propio Platt. El 06 mostró variación temporal, una excepción económica favorable al árbol y subestimación en [500,1000). Conservamos la elección y declaramos esos límites; no buscamos otro calibrador para favorecer una tabla.

Antes de escribir el protocolo comprobamos los parámetros serializados, las 432 entradas, los seis árboles y la alineación de scores/acciones. Una congelación existente se verifica y se reutiliza; no se sobrescribe. Si cambian fuentes, artefactos o decisiones, el proceso se detiene para revisar la versión del experimento.


In [2]:
protocol = freeze_protocol(ROOT)
display(pd.read_csv(ROOT / AUDIT))
display(pd.read_csv(ROOT / DECISIONS))
print("Estado:", protocol["status"])
print("Protocolo:", PROTOCOL)


,check,status,detail
0,cadena_03_04_05_06,passed,"Fuentes, registros, salidas y artefactos verif..."
1,seleccion_y_adopcion,passed,LightGBM referencia coincide en parámetros y 4...
2,platt_propio,passed,Calibrador del modelo adoptado; origen en vali...
3,poblacion_validation,passed,42047 operaciones posteriores y 1517 fraudes
4,acciones_alineadas,passed,252282 filas operación/Ca verificadas
5,costos_reconciliados,passed,Subperiodos y perturbación base reproducen los...
6,test_sin_etiquetas,passed,Solo ID/tiempo/split: 84233 operaciones futuras
7,escenarios,passed,Seis Ca hipotéticos; ninguna estimación de gas...


,decision,value,basis
0,predictor,lightgbm_referencia,AP media interna; 04
1,preparacion,432 ordinal/imputada,Mismo ajuste train; 03/05
2,calibrador,Platt propio,Mejora probabilística; límites 05/06 declarados
3,politicas,Fija 0.5; BMR; árbol por Ca,Matriz original común
4,Ca,"1,5,10,20,50,100",Escenarios hipotéticos; bibliografía y sensibi...
5,evaluacion,Test temporal completo; sin reajuste,Identidad/tiempo congelados; etiquetas agregad...


Estado: frozen_for_academic_evaluation
Protocolo: results/tables/protocolo_final.json


## 3. Qué queda fijo en el predictor y en Platt

Conservamos 37 categóricas ordinales y 395 numéricas imputadas. Los vocabularios y medianas se aprendieron solo en train. No agregamos atributos ni recodificamos los doce id documentados. El monto para costos permanece original en float64.

Platt recibe una columna de logits, no las 432 entradas. El clipping [10⁻⁶,1−10⁻⁶] precede al logit y la transformación es `sigmoid(a × logit(p_raw) + b)`. Sus parámetros se ajustaron en las 42.046 operaciones tempranas; no se reajustan con validación posterior ni test. Las probabilidades raw se conservan para el diagnóstico probabilístico; las tres políticas principales mantienen Platt y los árboles de acción ya aprendidos.


In [3]:
display(pd.DataFrame([{"modelo": protocol["model"]["candidate"], **protocol["model"]["params"]}]))
display(pd.DataFrame([{
    "metodo": protocol["calibration"]["method"],
    "a": protocol["calibration"]["coefficient_a"],
    "b": protocol["calibration"]["intercept_b"],
    "epsilon": protocol["calibration"]["clip_epsilon"],
    "filas_ajuste_calibrador": protocol["calibration"]["fit_rows"],
}]))


,modelo,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,n_jobs,random_state,verbose
0,lightgbm_referencia,250,0.05,63,50,0.8,0.8,4,42,-1


,metodo,a,b,epsilon,filas_ajuste_calibrador
0,platt_on_clipped_logit,1.044229,0.27463,0.000001,42046


## 4. Escenarios hipotéticos y políticas

Mantenemos Ca = 1, 5, 10, 20, 50 y 100. Los extremos 1 y 5 tienen antecedentes numéricos en la sensibilidad de [Bahnsen et al. (2013)](https://albahnsen.github.io/files/Cost%20Sensitive%20Credit%20Card%20Fraud%20Detection%20using%20Bayes%20Minimum%20Risk%20-%20Publish.pdf). El valor 10 aparece en [Bahnsen et al. (2014)](https://albahnsen.github.io/files/%20Improving%20Credit%20Card%20Fraud%20Detection%20by%20using%20Calibrated%20Probabilities%20-%20Publish.pdf). Esos trabajos usan euros y otras empresas: aquí solo inspiran escenarios, sin trasladar moneda ni estimar gastos IEEE-CIS. Los valores 20, 50 y 100 amplían la sensibilidad que ya veníamos estudiando.

El costo de intervenir es Ca en ambas clases; aprobar fraude cuesta A y aprobar legítima cuesta cero. La regla fija usa p≥0,5. BMR interviene si pA>Ca y aprueba en empate. El árbol recibe atributos y devuelve acciones; no recibe la probabilidad LightGBM. Comparamos las tres políticas en todas las operaciones de cada escenario. No elegimos Ca por el ahorro ni promediamos ahorros de escenarios con denominadores diferentes.


In [4]:
display(pd.DataFrame([
    {"estado": "fraude", "aprobar": "A original", "intervenir": "Ca"},
    {"estado": "legitima", "aprobar": "0", "intervenir": "Ca"},
]))
display(pd.DataFrame({"Ca_hipotetico": protocol["costs"]["ca_grid"],
                      "unidad": protocol["costs"]["unit"]}))
display(pd.DataFrame([{"politica": key, "regla": value} for key, value in protocol["policies"].items()]))


,estado,aprobar,intervenir
0,fraude,A original,Ca
1,legitima,0,Ca


,Ca_hipotetico,unidad
0,1,unidades de TransactionAmt; moneda no verificada
1,5,unidades de TransactionAmt; moneda no verificada
2,10,unidades de TransactionAmt; moneda no verificada
3,20,unidades de TransactionAmt; moneda no verificada
4,50,unidades de TransactionAmt; moneda no verificada
5,100,unidades de TransactionAmt; moneda no verificada


,politica,regla
0,fixed_0_5,p_platt >= 0.5
1,bmr,p_platt * TransactionAmt > Ca; ties approve
2,cost_sensitive_tree,reuse_stage03_tree_per_Ca; action_1_intervenes


## 5. Población futura y plan de resultados

La identidad de test se congela usando exclusivamente TransactionID, TransactionDT y split del manifiesto. No leemos sus etiquetas ni calculamos probabilidades. El bloque debe ser posterior a validación y conservar las 84.233 operaciones asignadas en el 02.

La evaluación posterior reportará costo total, intervención y fraude aprobado por política/Ca, ahorro frente a la mejor referencia trivial y mejora frente al fijo. Acompañará esos resultados con AP, Brier, log-loss y las métricas a 0,5 de raw/Platt; para acciones, conteos TP/FP/FN/TN, recall, intervención de legítimas y recall por monto. AP será `average_precision_score`, sin renombrarla como área trapezoidal. Los análisis por tres periodos y bandas de monto serán descriptivos, sin reajustar decisiones.


In [5]:
ready = verify_before_test(ROOT)
display(pd.DataFrame([ready["test"]]))
display(pd.DataFrame([{"familia": key, "contenido": str(value)} for key, value in ready["metrics"].items()]))
print("Identidad temporal verificada; etiquetas y predicciones test pendientes.")


,rows,time_start,time_end,id_time_sha256,identity_order,labels_read
0,84233,13270993,15811131,c77b2c456a3040f52b7302f57221099849cbfd1765e85e...,TransactionDT_then_TransactionID; little_endia...,False


,familia,contenido
0,primary,"['cost_total_proxy', 'administrative_cost_tota..."
1,probabilistic_raw_and_platt,"['average_precision', 'brier_score', 'log_loss..."
2,policy_diagnostics,"['TP', 'FP', 'FN', 'TN', 'fraud_recall', 'legi..."
3,average_precision_definition,sklearn.average_precision_score; not_trapezoid...
4,reference,"min(sum(y * original_amount), N * Ca)"
5,savings_reference,1 - cost/reference; NaN_if_reference_zero
6,improvement_fixed,1 - cost/fixed_cost; NaN_if_fixed_cost_zero
7,zero_precision,0_if_no_interventions
8,zero_class_denominator,NaN_if_class_absent


Identidad temporal verificada; etiquetas y predicciones test pendientes.


## 6. Interpretación y continuación

La evaluación tendrá carácter temporal, condicionado por la consulta histórica de etiquetas agregadas de test y por las decisiones de desarrollo que ya usaron validación posterior. No la describiremos como una reserva completamente ciega. La calibración no es perfecta y los costos suponen intervención eficaz, sin capacidad limitada, recuperación ni daño comercial modelados.

La siguiente etapa será la evaluación temporal final bajo este protocolo. Antes de puntuar deberá llamar `verify_before_test`, registrar el inicio con la huella del protocolo y conservar una ejecución auditable. Un registro existente impedirá repetir silenciosamente la evaluación. Una interrupción exige revisar y recuperar esa misma corrida, no cambiar el método tras observar sus resultados.

Los flags `final_protocol_frozen=False` de 05/06 describen su momento histórico. El registro posterior de esta etapa acredita la congelación técnica. El aval docente final y los costos de negocio no se inventan a partir de ese estado.


In [6]:
checked = verify_frozen_protocol(ROOT)
print("Congelación técnica verificada:", checked["status"])
print("SHA-256 del protocolo:", sha256_file(ROOT / PROTOCOL))
print("Registro:", RECORD)
print("Test evaluado en esta etapa:", checked["test_evaluated"])


Congelación técnica verificada: frozen_for_academic_evaluation
SHA-256 del protocolo: 0c4466ad14592ec3997c97bde8e577ac9aedb6b548555bf71dcca787ad293bbd
Registro: results/tables/protocolo_final_config.json
Test evaluado en esta etapa: False
